In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Etapy Transpilera

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Ta strona opisuje etapy gotowego potoku transpilacji w Qiskit SDK. Wyróżniamy sześć etapów:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

Funkcja [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) tworzy gotowy [staged pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) złożony z tych etapów. Konkretne przejścia składające się na każdy etap zależą od argumentów przekazanych do `generate_preset_pass_manager`. Argument `optimization_level` jest argumentem pozycyjnym, który musi być podany; jest to liczba całkowita, która może przyjmować wartość 0, 1, 2 lub 3. Wyższe wartości oznaczają bardziej kosztowną, lecz intensywniejszą optymalizację (zob. [Domyślne ustawienia transpilacji i opcje konfiguracji](defaults-and-configuration-options)).

Zalecanym sposobem transpilacji Circuit jest stworzenie gotowego staged pass managera, a następnie uruchomienie go na Circuit, zgodnie z opisem w [Transpilacja z pass managerami](transpile-with-pass-managers). Prostszą, choć mniej konfigurowalną alternatywą jest użycie funkcji [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). Funkcja ta przyjmuje Circuit bezpośrednio jako argument. Podobnie jak w przypadku `generate_preset_pass_manager`, konkretne przejścia Transpilera zależą od argumentów, takich jak `optimization_level`, przekazanych do `transpile`. W rzeczywistości wewnętrznie funkcja `transpile` wywołuje `generate_preset_pass_manager`, aby stworzyć gotowy staged pass manager, i uruchamia go na Circuit.

## Etap Init
Ten pierwszy etap domyślnie robi bardzo niewiele i jest przydatny głównie wtedy, gdy chcesz uwzględnić własne wstępne optymalizacje. Ponieważ większość algorytmów układu i routingu jest zaprojektowana do pracy wyłącznie z bramkami jednoQubitowymi i dwuQubitowymi, ten etap służy również do przekształcania wszelkich Gate działających na więcej niż dwóch Qubitach w Gate działające tylko na jednym lub dwóch Qubitach.

Więcej informacji na temat implementacji własnych wstępnych optymalizacji dla tego etapu znajdziesz w sekcji dotyczącej wtyczek i dostosowywania pass managerów.

## Etap Layout
Kolejny etap dotyczy układu lub łączności Backendu, do którego Circuit zostanie wysłany. Ogólnie rzecz biorąc, Circuit kwantowe są abstrakcyjnymi bytami, których Qubity są „wirtualnymi" lub „logicznymi" reprezentacjami rzeczywistych Qubitów używanych w obliczeniach. Aby wykonać sekwencję Gate, niezbędne jest odwzorowanie jeden do jednego „wirtualnych" Qubitów na „fizyczne" Qubity rzeczywistego urządzenia kwantowego. To odwzorowanie jest przechowywane jako obiekt `Layout` i stanowi część ograniczeń zdefiniowanych w [architekturze zestawu instrukcji (ISA)](./transpile#instruction-set-architecture) Backendu.

![Ten obraz ilustruje mapowanie Qubitów z reprezentacji przewodów na diagram przedstawiający sposób połączenia Qubitów w QPU.](../docs/images/guides/transpiler-stages/layout-mapping.avif "Mapowanie Qubitów")

Wybór odwzorowania ma ogromne znaczenie dla zminimalizowania liczby operacji SWAP potrzebnych do zmapowania wejściowego Circuit na topologię urządzenia oraz zapewnienia, że używane są najlepiej skalibrowane Qubity. Ze względu na wagę tego etapu, gotowe pass managery wypróbowują kilka różnych metod, aby znaleźć najlepszy layout. Zazwyczaj obejmuje to dwa kroki: najpierw próba znalezienia „idealnego" layoutu (layoutu niewymagającego żadnych operacji SWAP), a następnie heurystyczne przejście próbujące znaleźć najlepszy layout do użycia, gdy idealny layout nie może zostać znaleziony. Na tym pierwszym etapie typowo używane są dwa `Passes`:

- `TrivialLayout`: Naiwnie odwzorowuje każdy wirtualny Qubit na fizyczny Qubit o tym samym numerze w urządzeniu (tzn. [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]). Jest to historyczne zachowanie używane wyłącznie w `optimization_level=1` w celu znalezienia idealnego layoutu. Jeśli się to nie powiedzie, następnie próbowany jest `VF2Layout`.
- `VF2Layout`: Jest to `AnalysisPass`, który wybiera idealny layout, traktując ten etap jako problem izomorfizmu podgrafów, rozwiązywany przez algorytm VF2++. Jeśli zostanie znaleziony więcej niż jeden layout, uruchamiana jest heurystyczna funkcja oceniająca, aby wybrać odwzorowanie z najniższym średnim błędem.

Następnie w etapie heurystycznym domyślnie używane są dwa przejścia:

- `DenseLayout`: Znajduje podgraf urządzenia o największej łączności, posiadający tę samą liczbę Qubitów co Circuit (używany dla poziomu optymalizacji 1, jeśli w Circuit są obecne operacje przepływu sterowania, takie jak `IfElseOp`).
- `SabreLayout`: To przejście wybiera layout, zaczynając od losowego układu początkowego i wielokrotnie uruchamiając algorytm `SabreSwap`. Przejście to jest używane tylko na poziomach optymalizacji 1, 2 i 3, gdy za pomocą przejścia `VF2Layout` nie zostanie znaleziony idealny layout. Więcej szczegółów dotyczących tego algorytmu znajdziesz w artykule [arXiv:1809.02573.](https://arxiv.org/abs/1809.02573)

## Etap Routing
Aby zaimplementować dwuQubitową Gate między Qubitami, które nie są bezpośrednio połączone w urządzeniu kwantowym, do Circuit należy wstawić jedną lub więcej Gate SWAP w celu przemieszczenia stanów Qubitów, aż znajdą się sąsiadujące na mapie Gate urządzenia. Każda Gate SWAP oznacza kosztowną i podatną na szumy operację do wykonania. Dlatego znalezienie minimalnej liczby Gate SWAP potrzebnych do zmapowania Circuit na dane urządzenie jest ważnym krokiem w procesie transpilacji. Dla wydajności ten etap jest domyślnie zazwyczaj obliczany wspólnie z etapem Layout, choć są one od siebie logicznie odrębne. Etap *Layout* wybiera fizyczne Qubity do użycia, natomiast etap *Routing* wstawia odpowiednią liczbę Gate SWAP w celu wykonania Circuit przy użyciu wybranego layoutu.

Znalezienie optymalnego mapowania SWAP jest jednak trudne. Jest to w rzeczywistości problem NP-trudny, przez co jego obliczanie jest kosztowne praktycznie dla wszystkich poza najmniejszymi urządzeniami kwantowymi i wejściowymi Circuit. Aby to obejść, Qiskit używa stochastycznego algorytmu heurystycznego o nazwie `SabreSwap` do obliczania dobrego, choć niekoniecznie optymalnego mapowania SWAP. Użycie metody stochastycznej oznacza, że generowane Circuit nie muszą być takie same przy powtarzanych uruchomieniach. Rzeczywiście, wielokrotne uruchamianie tego samego Circuit prowadzi do rozkładu głębokości Circuit i liczby Gate na wyjściu. Z tego powodu wielu użytkowników decyduje się na wielokrotne uruchamianie funkcji routingu (lub całego `StagedPassManager`) i wybieranie Circuit o najmniejszej głębokości z rozkładu wyników.

Na przykład przyjrzyjmy się 15-Qubitowemu obwodowi GHZ wykonanemu 100 razy, korzystając ze „złego" (rozłączonego) `initial_layout`.

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_ibm_runtime.fake_provider import FakeAuckland, FakeWashingtonV2
from qiskit.transpiler import generate_preset_pass_manager

backend = FakeAuckland()

ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/358cfb50-02fc-48f2-a4ec-657837e0c304-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/bb3b8c1f-69fd-4e0c-9b78-9ee67f3361bb-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](./transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'for_loop', 'id', 'if_else', 'measure', 'reset', 'rz', 'switch_case', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

Jak widać, ten Circuit musi wykonać dwuQubitową Gate między Qubitami 0 i 14, które są od siebie bardzo oddalone na grafie łączności. Uruchomienie tego Circuit wymaga zatem wstawiania Gate SWAP w celu wykonania wszystkich dwuQubitowych Gate przy użyciu przejścia `SabreSwap`.

Zauważ również, że algorytm `SabreSwap` różni się od szerszej metody `SabreLayout` z poprzedniego etapu. Domyślnie `SabreLayout` wykonuje zarówno layout, jak i routing, i zwraca przekształcony Circuit. Jest to spowodowane kilkoma szczególnymi względami technicznymi opisanymi na [stronie dokumentacji API](../api/qiskit/qiskit.transpiler.passes.SabreLayout) przejścia.

## Etap Translation
Pisząc Circuit kwantowy, możesz swobodnie używać dowolnej kwantowej Gate (operacji unitarnej), a także zbioru operacji innych niż Gate, takich jak instrukcje pomiaru lub resetowania Qubitu. Jednak większość urządzeń kwantowych natywnie obsługuje tylko niewielką liczbę kwantowych Gate i operacji innych niż Gate. Te natywne Gate są częścią definicji [ISA](./transpile#instruction-set-architecture) docelowego urządzenia i ten etap gotowych `PassManagerów` przekształca (lub *rozwijania*) Gate określone w Circuit na natywne Gate bazowe określonego Backendu. Jest to ważny krok, ponieważ umożliwia wykonanie Circuit przez Backend, ale zazwyczaj prowadzi do zwiększenia głębokości i liczby Gate.

Warto podkreślić dwa szczególne przypadki, które dobrze ilustrują, co robi ten etap.

1. Jeśli Gate SWAP nie jest natywną Gate docelowego Backendu, wymaga to trzech Gate CNOT:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2507de9c-1b94-4d56-b5a6-0b2bb1719a80-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

Jako iloczyn trzech Gate CNOT, SWAP jest kosztowną operacją do wykonania na szumiących urządzeniach kwantowych. Takie operacje są jednak zazwyczaj niezbędne do osadzenia Circuit w ograniczonych łącznościach Gate wielu urządzeń. Minimalizacja liczby Gate SWAP w Circuit jest zatem głównym celem procesu transpilacji.

2. Toffoli, czyli bramka controlled-controlled-not (`ccx`), jest trzuQubitową Gate. Biorąc pod uwagę, że nasz bazowy zestaw Gate zawiera tylko Gate jednoQubitowe i dwuQubitowe, operacja ta musi zostać zdekomponowana. Jest to jednak dość kosztowne:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = FakeWashingtonV2()

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4339deb5-4947-493b-8940-e68c91311631-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

Na każdą Gate Toffoli w Circuit kwantowym sprzęt może wykonać do sześciu Gate CNOT i kilka Gate jednoQubitowych. Ten przykład pokazuje, że każdy algorytm korzystający z wielu Gate Toffoli stanie się Circuit o dużej głębokości i będzie zatem w znacznym stopniu dotknięty przez szumy.

## Etap Optimization
Ten etap skupia się na dekompozycji Circuit kwantowych do zestawu Gate bazowych docelowego urządzenia i musi zwalczać zwiększoną głębokość wynikającą z etapów layout i routing. Na szczęście istnieje wiele procedur optymalizacji Circuit poprzez łączenie lub eliminowanie Gate. W niektórych przypadkach metody te są tak skuteczne, że wyjściowe Circuit mają mniejszą głębokość niż wejściowe, nawet po layoutcie i routingu do topologii sprzętowej. W innych przypadkach nie wiele można zrobić, a obliczenia mogą być trudne do wykonania na szumiących urządzeniach. Na tym etapie poszczególne poziomy optymalizacji zaczynają się od siebie różnić.

- Dla `optimization_level=1` ten etap przygotowuje [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) i [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), które łączą łańcuchy jednoQubitowych Gate i anulują kolejne Gate CNOT.
- Dla `optimization_level=2` ten etap zamiast `CXCancellation` używa przejścia [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation), które usuwa nadmiarowe Gate, korzystając z relacji przemienny.
- Dla `optimization_level=3` ten etap przygotowuje następujące przejścia:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

Ponadto ten etap wykonuje również kilka końcowych sprawdzeń, aby upewnić się, że wszystkie instrukcje w Circuit są złożone z Gate bazowych dostępnych w docelowym Backendzie.

Poniższy przykład ze stanem GHZ demonstruje efekty różnych ustawień poziomu optymalizacji na głębokość Circuit i liczbę Gate.

> **Note:** Wynik transpilacji różni się ze względu na stochastyczny mapper SWAP. Dlatego poniższe liczby będą się prawdopodobnie zmieniać przy każdym uruchomieniu kodu.

![15-Qubitowy stan GHZ](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-Qubitowy stan GHZ przed transpilacją")

Poniższy kod konstruuje 15-Qubitowy stan GHZ i porównuje `optimization_levels` transpilacji pod względem wynikowej głębokości Circuit, liczby Gate i liczby wieloQubitowych Gate.